# Binary B model (prototype)

Julia version of the C model of Melo & Marroig (2015): binary ontogenetic matrix `b`,
`z = b * y + e`. A population evolves under drift and stabilizing selection, then under
divergent directional selection on two modules (traits `1:p/2` up, `p/2+1:p` down).
See `?EvoMod.BinaryB` for the model.

In [ ]:
using Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

using EvoMod, EvoMod.BinaryB
using Gadfly, DataFrames, LinearAlgebra, Random, Statistics
include("plots.jl")
Random.seed!(42)

In [ ]:
# Parameters
n_e  = 5000
m    = 400        # loci
p    = 8          # traits
v_e  = 0.8        # environmental standard deviation
mu   = 0.0005     # mutation rate of allelic effects
mu_b = 0.0001     # mutation rate of b
mu_σ = 0.02       # standard deviation of mutational effects

omega_var = 10
delta_theta_speed = 20/1000

burnin_generations = [100, 100]   # Drift, Stab
direct_generations = [500, 500]   # Direct, Stab
thin = 1                          # record moments every `thin` generations

ind = Individual(m, p, v_e, mu, mu_b, mu_σ)

In [ ]:
diag_omega = omega_var * Matrix(1.0I, p, p)
burnin_pop = run_pop(ind, n_e, ["Drift", "Stab"], burnin_generations; omega = diag_omega, thin = thin)

In [ ]:
delta_theta = delta_theta_speed * [ones(p ÷ 2); -ones(p ÷ 2)]
omega = twoModuleMatrix(omega_var * ones(p), 0.8, 0.8)

direct_pop = run_pop(copy(burnin_pop), ["Direct", "Stab"], direct_generations;
                     delta_theta = delta_theta, omega = omega, thin = thin)
pop = direct_pop

In [ ]:
plotCorrHeatmap(pop.moments[end]["corrG"])

In [ ]:
plotCorrelations(pop)

In [ ]:
plotSurfacePop(pop, pop.surface)

In [ ]:
plotTraits(pop)

Distance between the mean phenotype and the optimum along the two main axes of the fitness surface:

In [ ]:
V = eigen(Symmetric(cov(pop.surface))).vectors[:, (end - 1):end]
V' * pop.moments[end]["mean_z"] - V' * mean(pop.surface)

In [ ]:
plotMoment(pop, "h2")